# w10a — Decision Errors, Power, and Classification Metrics
**ENGR 1451/2451 · Exploratory Data Science · Fall 2026**

This short Python notebook uses the same fair/loaded dice, significance thresholds, base-rate example, and manufacturing strength data as the slides.

We'll use `scipy.stats` to calculate probabilities and cutoffs, `numpy` to repeat die experiments, and `pandas` to organize the results. Die-test calculations use the slides' **normal approximation**; exact probabilities can differ slightly because die totals are discrete.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 6)
alpha = 0.05
fair_mu, fair_sd = 3.5, np.sqrt(35 / 12)
loaded_mu, loaded_sd = 4.0, np.sqrt(10 / 3)

## 1. Same observed average, different numbers of rolls

The slides compare **100 rolls (368 spots)** with **1,000 rolls (3,680 spots)**. Both have observed average $\bar x=3.68$. Under the fair-die model,

$$z=\frac{\bar x-3.5}{\sigma_0/\sqrt{n}}, \qquad
P\text{-value}\approx 2P(Z\geq|z|).$$

`stats.norm.sf(x)` gives the standard-normal **upper-tail** probability. Compare each $P$-value with $\alpha=0.05$.

In [ ]:
roll_comparison = pd.DataFrame({"n": [100, 1000], "Observed mean": [3.68, 3.68]})
roll_comparison["SE"] = fair_sd / np.sqrt(roll_comparison["n"])
roll_comparison["z"] = (roll_comparison["Observed mean"] - fair_mu) / roll_comparison["SE"]
roll_comparison["Two-sided p"] = 2 * stats.norm.sf(abs(roll_comparison["z"]))
roll_comparison["Flag at 5%?"] = roll_comparison["Two-sided p"] < alpha
roll_comparison

## 2. One rule, two errors

Our specified loaded die has $P(6)=1/3$ and $P(1)=\cdots=P(5)=2/15$, so its **mean is 4.0**. The same decision rule applies whether a die is actually fair or loaded.

For $n=100$, the fair and loaded **sample means** have approximate normal sampling distributions with standard deviations $\sigma_0/\sqrt n$ and $\sigma_1/\sqrt n$.

- `model.ppf(q)` finds a cutoff with lower-tail probability `q`.
- `model.cdf(upper) - model.cdf(lower)` finds the chance of falling **between** two cutoffs.

$$\alpha=P(\mathrm{flag}\mid\mathrm{fair}),\qquad
\beta=P(\mathrm{not\ flag}\mid\mathrm{loaded}),\qquad
\mathrm{Power}=1-\beta.$$

In [ ]:
n_rolls = 100
fair_model = stats.norm(loc=fair_mu, scale=fair_sd / np.sqrt(n_rolls))
loaded_model = stats.norm(loc=loaded_mu, scale=loaded_sd / np.sqrt(n_rolls))

lower, upper = fair_model.ppf([alpha / 2, 1 - alpha / 2])
false_positive_rate = fair_model.cdf(lower) + fair_model.sf(upper)
beta = loaded_model.cdf(upper) - loaded_model.cdf(lower)
power = 1 - beta

print(f"Flag if sample mean < {lower:.3f} or > {upper:.3f}")
print(f"Alpha: {false_positive_rate:.3f}  |  Beta: {beta:.3f}  |  Power: {power:.3f}")

## 3. Tighten the cutoff or collect more data?

The slides show that **smaller $\alpha$** reduces false alarms but also reduces power at fixed $n$. More rolls can recover power while keeping the same $\alpha$.

Use `stats.norm.ppf()` for each plan's cutoffs, then the *loaded* sampling distribution to calculate $\beta$. This is still the **same specified type of loaded die**.

*Numbers displayed as 100% power in the table are rounded; no finite-sample test here has exactly perfect power.*

In [ ]:
plans = pd.DataFrame({
    "Rolls": [100, 100, 100, 100, 400, 1000],
    "alpha": [0.05, 0.01, 0.001, 0.0001, 0.05, 0.001],
})

z_star = stats.norm.ppf(1 - plans["alpha"].to_numpy() / 2)
se0 = fair_sd / np.sqrt(plans["Rolls"].to_numpy())
se1 = loaded_sd / np.sqrt(plans["Rolls"].to_numpy())
lo, hi = fair_mu - z_star * se0, fair_mu + z_star * se0
beta_plans = stats.norm.cdf(hi, loaded_mu, se1) - stats.norm.cdf(lo, loaded_mu, se1)

plans["alpha (%)"] = 100 * plans.pop("alpha")
plans["beta (%)"] = 100 * beta_plans
plans["power (%)"] = 100 * (1 - beta_plans)
plans[["Rolls", "alpha (%)", "beta (%)", "power (%)"]].round(3)

## 4. Rare loaded dice: positive predictive value (PPV)

The slides consider **1,000,000 dice**, including **1,000 loaded (0.1%)**. Use their rounded rates: **82% sensitivity** and **95% specificity**. Create a two-way table of expected counts.

$$\mathrm{PPV}=P(\mathrm{loaded}\mid\mathrm{flagged})=\frac{TP}{TP+FP},\qquad
\mathrm{Accuracy}=\frac{TP+TN}{N}.$$

**PPV (positive predictive value) and sensitivity are different conditional probabilities.** PPV is also called *precision* in machine learning, but that term is distinct from **measurement precision**.

In [ ]:
n_total, n_loaded = 1_000_000, 1_000
n_fair = n_total - n_loaded
sensitivity, specificity = 0.82, 0.95  # rounded rates from slides

TP = round(n_loaded * sensitivity)
FN = n_loaded - TP
TN = round(n_fair * specificity)
FP = n_fair - TN


counts = pd.DataFrame({
    "Do not flag": [TN, FN],
    "Flag": [FP, TP],
}, index=["Fair", "Loaded"])

counts.index.name = "Actual condition"
display(counts)

ppv = TP / (TP + FP)
accuracy = (TP + TN) / n_total
print(f"PPV (loaded | flagged): {ppv:.2%}")
print(f"Screening accuracy:    {accuracy:.2%}")
print(f"Always say 'fair':     {n_fair / n_total:.2%} accuracy, 0% sensitivity")


## 5. Material strength: use Student's $t$ when $\sigma$ is unknown

The slides give a **hypothetical sample**, not raw measurements: $n=25$, $\bar x=48$ MPa, and **sample SD** $s=5$ MPa. Test $H_0:\mu\geq50$ against $H_A:\mu<50$.

$$\widehat{\mathrm{SE}}=\frac{s}{\sqrt n},\qquad
t=\frac{\bar x-50}{\widehat{\mathrm{SE}}},\qquad df=n-1.$$

Because this is a **lower-tailed** test, use `stats.t.cdf()` for its $P$-value. The sample does **not** tell us $\beta$: that would also require specifying an alternative true mean and a population variability model.

In [ ]:
strength_n = 25
strength_mean, strength_s = 48.0, 5.0  # sample mean and sample SD (not known sigma)
strength_se = strength_s / np.sqrt(strength_n)
t_stat = (strength_mean - 50) / strength_se

df = strength_n - 1
p_value = stats.t.cdf(t_stat, df=df)
t_cutoff = stats.t.ppf(alpha, df=df)

print(f"Estimated SE: {strength_se:.3f} MPa")
print(f"t = {t_stat:.3f} (df={df});  one-sided p = {p_value:.4f}")
print(f"Critical t = {t_cutoff:.3f}")
print("Decision:", "Flag the batch" if p_value < alpha else "Do not flag")

### Python function reference

| Function or method | General use |
|---|---|
| `stats.norm(loc=mean, scale=sd)` | Define a normal distribution. |
| `model.ppf(q)` | Find the value whose lower-tail probability is `q`. |
| `model.cdf(x)` / `model.sf(x)` | Probability to the left / right of `x`. |
| `rng.choice(values, size=shape, p=probs)` | Draw random values; `p` is optional. |
| `array.mean(axis=1)` | Compute each row's mean. |
| `np.mean(boolean_array)` | Find the fraction of `True` values. |
| `pd.DataFrame(data)` | Organize results into a table. |
| `stats.t.cdf(t_stat, df=df)` | Lower-tail $t$ probability. |

## Your turn — a stricter screening plan

A factory tests each die **400 times** at a two-sided significance level of **$\alpha=0.001$ (0.1%)**. Use the fair and specified loaded-die models from Section 2.

1. Compute the two cutoffs using `stats.norm(...).ppf()`.
2. Calculate $\beta$ and power using the **loaded** sampling distribution.
3. If 1,000 of one million dice are loaded, find the expected **true-positive** and **false-positive** counts and the **PPV**.
4. Explain in one sentence why 99.9% specificity does *not* mean 99.9% PPV.

*Use the normal approximation; round expected counts only when reporting them.*

In [ ]:
new_n, new_alpha = 400, 0.001
loaded_fraction = 0.001
n_factory = 1_000_000

# 1. Define the fair/loaded distributions and obtain cutoffs.
# YOUR CODE HERE

# 2. Compute beta and power.
# YOUR CODE HERE

# 3. Compute expected true/false flags and PPV.
# YOUR CODE HERE

# 4. In one sentence: Why is specificity different from PPV?
# YOUR ANSWER: